# 🧠 BCI Competition IV — Dataset 2a · Motor Imagery Yeniden-Üretim Notebook'u

**4 sınıflı motor imagery** (sol el / sağ el / iki ayak / dil) EEG sınıflandırması.
Bu notebook, çok-dosyalı araştırma projesini **tek bir Colab notebook'una** indirger;
`Runtime ▸ Run all` ile sıfırdan, baştan sona çalıştırıldığında aynı final sonucu üretir.

### 🎯 Beklenen final sonuç (saklı test seti A0XE)

| Metrik | Değer |
|:--|:--|
| **Final Test Cohen's κ** | **0.6137** |
| Final Test accuracy | 0.7103 |
| Δ (Test − CV) | −0.042 (makul session shift, leakage yok) |
| Yarışma kazananı (Kai Keng Ang, FBCSP, 2008) | κ = 0.569 → **+0.045 geçildi** |

> ℹ️ Klasik fazlar (FBCSP, Riemann) tamamen deterministiktir → κ bit-bit yeniden üretilir.
> Derin öğrenme fazları (EEGNet / ShallowConvNet) GPU çekirdek non-determinizmi nedeniyle
> ±0.01–0.02 oynayabilir; tüm tohumlar sabitlendiği için sapma küçüktür. Her bölüm
> **elde edilen κ'yı beklenen değerle karşılaştırarak** bilgilendirir (assert yok).

### 🔒 En kritik ilke — LEAKAGE-FREE
Test seti **A0XE, BÖLÜM 7'ye kadar HİÇ kullanılmaz.** Tüm hiperparametre seçimi iç
cross-validation'da yapılır. Tüm fazlar **`StratifiedKFold(5, shuffle=True, random_state=42)`**
kullanır → fold yapısı her fazda bit-bit aynı → OOF tahminleri trial-by-trial hizalı →
ensemble temiz.

### ⏱️ Tahmini toplam süre (Colab T4)
| Bölüm | Tahmin |
|:--|:--|
| 0–1 Kurulum + veri indirme | ~5 dk |
| 3 Faz 1 (FBCSP+sLDA) | ~30–80 dk (en yavaş klasik) |
| 4 Faz 2–3 ablasyonlar | ~40–70 dk |
| 6 Faz 5 (EEGNet + ShallowConvNet) | ~15–20 dk (GPU) |
| 7 Faz 6 (final test, 9 denek) | ~25–35 dk |
| **TOPLAM** | **~2–4 saat** |

> 🧪 **Hızlı deneme:** Bölüm 0'daki `SUBJECTS` listesini örn. `[1, 3]` yaparak yapıyı
> dakikalar içinde sınayabilirsiniz (κ değerleri 9-denek ortalamasına göre verilmiştir).

### 📑 İçindekiler
0. Kurulum (paketler, GPU, tohum)
1. Veri indirme + yükleme + ön işleme
2. Değerlendirme altyapısı (nested CV, OOF, soft-vote)
3. **Faz 1** — Baseline: Regularized FBCSP + Shrinkage-LDA (κ≈0.625)
4. **Faz 2–3** — Ablasyon / negatif sonuçlar (band-sel, Riemann)
5. **Faz 4** — Klasik ensemble (κ≈0.6425)
6. **Faz 5** — Derin öğrenme + final ensemble (κ≈0.6553)
7. **Faz 6** — A0XE saklı test, tek-shot (**Test κ=0.6137**)
8. Görselleştirme & özet

## BÖLÜM 0 — Kurulum

Gerekli paketleri kuruyoruz. Colab'da `numpy`, `scipy`, `scikit-learn`, `torch`,
`matplotlib`, `joblib` zaten yüklüdür; **`mne`** (GDF okuma) ve **`pyriemann`**
(Riemann geometrisi) ek olarak kurulur.

In [ ]:
# ~1-2 dk. mne: .gdf okuma; pyriemann: kovaryans manifoldu yöntemleri.
!pip -q install -U mne pyriemann
print("Kurulum tamam.")

In [ ]:
# --- İmportlar, GPU kontrolü, global yapılandırma ---
import os, sys, time, ssl, shutil, zipfile, urllib.request, warnings
from pathlib import Path
from functools import partial

import numpy as np
import torch

warnings.filterwarnings("ignore")

# GPU kontrolü (DL fazları için). Yoksa CPU'da çalışır ama yavaştır.
if torch.cuda.is_available():
    print("✅ GPU bulundu:", torch.cuda.get_device_name(0))
else:
    print("⚠️  GPU YOK — derin öğrenme fazları CPU'da çok yavaş olacaktır.")
    print("    Colab'da: Runtime ▸ Change runtime type ▸ Hardware accelerator ▸ T4 GPU")

# ---- Global yapılandırma ----
DATA_DIR_PATH = Path("/content/data")          # .gdf dosyaları buraya
LABELS_DIR    = DATA_DIR_PATH / "true_labels"   # A0XE.mat etiketleri buraya
SUBJECTS      = list(range(1, 10))              # tüm 9 denek (hızlı deneme: [1, 3])
RUN_TRANSFER  = False                           # Faz 5b LOSO transfer (opsiyonel, ~50 dk)

# Determinizm — tüm random_state=42
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print("Denekler:", SUBJECTS)
print("Veri dizini:", DATA_DIR_PATH)

## BÖLÜM 1 — Veri İndirme, Yükleme ve Ön İşleme

### 1.1 Otomatik indirme

İki kaynaktan veri çekiyoruz:

1. **`.gdf` ham EEG** — BNCI Horizon 2020 aynası (MOABB'nin de kullandığı kaynak):
   `bnci-horizon-2020.eu/database/data-sets/001-2014/A0X{T,E}.gdf` (18 dosya, ~‑40 MB/dosya).
   Bu ayna kayıt/lisans gerektirmez ve doğrudan indirilebilir (TU‑Graz sunucusuna yönlenir).
2. **Gerçek test etiketleri (`.mat`)** — resmi yarışma sonuçları:
   `bbci.de/competition/iv/results/ds2a/true_labels.zip` → `A0XE.mat` (test etiketleri).

> 🛟 **Manuel alternatif (indirme başarısızsa):** Dosyaları Google Drive'a yükleyip
> `DATA_DIR_PATH`'i Drive yolunuza işaret edin, ardından bu hücreyi atlayıp 1.3'e geçin.
> Gerekli düzen: `DATA_DIR_PATH/A0X{T,E}.gdf` ve `DATA_DIR_PATH/true_labels/A0XE.mat`.

In [ ]:
DATA_DIR_PATH.mkdir(parents=True, exist_ok=True)
LABELS_DIR.mkdir(parents=True, exist_ok=True)

GDF_BASE = "https://bnci-horizon-2020.eu/database/data-sets/001-2014/"
LABELS_URL = "https://www.bbci.de/competition/iv/results/ds2a/true_labels.zip"

def _download(url, dst, min_bytes=10000, tries=3):
    if dst.exists() and dst.stat().st_size > min_bytes:
        print("  (var)", dst.name); return
    ctx = ssl.create_default_context()
    ctx.check_hostname = False; ctx.verify_mode = ssl.CERT_NONE
    last = None
    for t in range(tries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, context=ctx, timeout=180) as r, open(dst, "wb") as f:
                shutil.copyfileobj(r, f)
            print("  ✓", dst.name, f"({dst.stat().st_size/1e6:.1f} MB)"); return
        except Exception as e:
            last = e; print(f"  ! deneme {t+1} hata: {e}")
    raise RuntimeError(f"İndirilemedi: {url}\n{last}")

print("== .gdf dosyaları ==")
for sid in range(1, 10):
    for sess in ("T", "E"):
        _download(GDF_BASE + f"A{sid:02d}{sess}.gdf", DATA_DIR_PATH / f"A{sid:02d}{sess}.gdf")

print("\n== gerçek test etiketleri (.mat) ==")
zp = DATA_DIR_PATH / "true_labels.zip"
_download(LABELS_URL, zp, min_bytes=1000)
with zipfile.ZipFile(zp) as z:
    z.extractall(LABELS_DIR)
# Olası alt-klasörleri düzleştir → true_labels/A0XE.mat
for p in list(LABELS_DIR.rglob("*.mat")):
    if p.parent != LABELS_DIR:
        shutil.move(str(p), str(LABELS_DIR / p.name))
print("Etiket dosyaları:", sorted(p.name for p in LABELS_DIR.glob("A0*E.mat")))

### 1.2 Proje kütüphanesi (`bci_lib.py`)

Tüm yeniden kullanılabilir mantığı (veri yükleyici, değerlendirme, FBCSP/Riemann/DL
yapı taşları, modeller) **importlanabilir bir modüle** yazıyoruz.

**Neden ayrı modül?** `GridSearchCV(n_jobs=-1)` ve `joblib.Memory`, özel transformer
sınıflarını paralel alt-süreçlere *pickle* eder. Notebook hücresinde (`__main__`)
tanımlanan sınıflar bu işlemde sorun çıkarır; modül yolundan import edilen sınıflar
sorunsuz çalışır. **Mantık, orijinal çok-dosyalı projeden birebir taşınmıştır.**

Aşağıdaki hücre modülü diske yazar (içeriği tamamen görünürdür — eğitici amaçla okunabilir).

In [ ]:
%%writefile bci_lib.py
"""
bci_lib.py  —  BCI Competition IV 2a yeniden-üretim kütüphanesi
================================================================

Bu modül, projedeki tüm yeniden kullanılabilir yapı taşlarını TEK bir
importlanabilir dosyada toplar. Notebook bu dosyayı `%%writefile` ile
çalışma zamanında diske yazar ve `import bci_lib` ile yükler.

Neden ayrı bir .py modülü?  GridSearchCV(n_jobs=-1) ve joblib.Memory, özel
transformer SINIFLARINI alt-süreçlere "pickle" eder. Notebook hücresinde
(__main__) tanımlanan sınıflar bu pickle işleminde sorun çıkarır; modül
yolundan import edilen sınıflar sorunsuz çalışır. Mantık birebir korunmuştur.

İçerik:
  - Veri yükleme (load_subject / load_all_subjects)        [data_loader.py]
  - Değerlendirme (run_nested_cv, soft_vote, metrikler)    [evaluation.py]
  - FBCSP yapı taşları (CSP, band-seçim, mRMR)             [_fbcsp_common.py]
  - Riemann yapı taşları (TS, multi-band TS)               [_riemann_common.py]
  - Derin öğrenme altyapısı (Dataset, train, transfer)     [_dl_common.py]
  - Modeller (EEGNet, ShallowConvNet)                      [exp_eegnet/scnet]
"""

from __future__ import annotations

import os
import time
import warnings
from dataclasses import dataclass, field
from functools import partial
from pathlib import Path
from typing import (Any, Callable, Dict, List, Optional, Sequence, Tuple)

import numpy as np

warnings.filterwarnings("ignore", category=RuntimeWarning, module="mne")
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
warnings.filterwarnings("ignore", category=RuntimeWarning, module="sklearn")
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# =========================================================================== #
# GENEL SABİTLER                                                              #
# =========================================================================== #

RANDOM_STATE = 42

#: Veri kök dizini. Notebook bunu bci_lib.DATA_DIR = ... ile değiştirir.
DATA_DIR = Path("/content/data")

SFREQ = 250.0
N_EEG_CHANNELS = 22
N_EOG_CHANNELS = 3

CUE_EVENT_IDS_TRAIN = {"769": 1, "770": 2, "771": 3, "772": 4}
CUE_EVENT_ID_TEST = {"783": 0}

TMIN_AFTER_CUE = 0.5
TMAX_AFTER_CUE = 2.5
EPOCH_SAMPLES = int(round((TMAX_AFTER_CUE - TMIN_AFTER_CUE) * SFREQ))  # 500
LABEL_OFFSET = 1


# =========================================================================== #
# VERİ YÜKLEME  (data_loader.py)                                              #
# =========================================================================== #


@dataclass
class SubjectData:
    """Tek bir oturumun epoch'lanmış verisi."""
    X: np.ndarray
    y: Optional[np.ndarray]
    X_eog: np.ndarray
    subject_id: int
    session: str
    sfreq: float
    ch_names: List[str]

    @property
    def n_trials(self) -> int:
        return self.X.shape[0]

    @property
    def class_distribution(self) -> Dict[int, int]:
        if self.y is None:
            return {}
        unique, counts = np.unique(self.y, return_counts=True)
        return {int(k): int(v) for k, v in zip(unique, counts)}


def _data_dir(data_dir: Optional[Path]) -> Path:
    return Path(data_dir) if data_dir is not None else DATA_DIR


def _build_gdf_path(subject_id: int, session: str, data_dir: Path) -> Path:
    if subject_id < 1 or subject_id > 9:
        raise ValueError(f"subject_id 1..9 olmalı, alınan: {subject_id}")
    session = session.upper()
    if session not in {"T", "E"}:
        raise ValueError(f"session 'T' veya 'E' olmalı, alınan: {session}")
    path = data_dir / f"A{subject_id:02d}{session}.gdf"
    if not path.exists():
        raise FileNotFoundError(f"GDF dosyası bulunamadı: {path}")
    return path


def _find_true_labels_file(subject_id: int, data_dir: Path) -> Optional[Path]:
    candidates = [
        data_dir / f"A{subject_id:02d}E.mat",
        data_dir / "true_labels" / f"A{subject_id:02d}E.mat",
        data_dir / "labels" / f"A{subject_id:02d}E.mat",
    ]
    for c in candidates:
        if c.exists():
            return c
    return None


def _load_true_labels(mat_path: Path) -> np.ndarray:
    from scipy.io import loadmat
    mat = loadmat(str(mat_path))
    if "classlabel" not in mat:
        raise KeyError(f"{mat_path.name} içinde 'classlabel' yok.")
    labels = mat["classlabel"].ravel().astype(int)
    return labels - LABEL_OFFSET  # 1..4 -> 0..3


def load_subject(
    subject_id: int,
    session: str = "T",
    data_dir: Optional[Path] = None,
    tmin: float = TMIN_AFTER_CUE,
    tmax: float = TMAX_AFTER_CUE,
    verbose: bool = False,
) -> SubjectData:
    """Tek deneğin tek oturumunu yükle ve epoch'la (288, 22, 500)."""
    import mne
    data_dir = _data_dir(data_dir)
    gdf_path = _build_gdf_path(subject_id, session, data_dir)
    mne_verbose = "INFO" if verbose else "ERROR"

    raw = mne.io.read_raw_gdf(str(gdf_path), preload=True, verbose=mne_verbose)

    ch_names_all = raw.ch_names
    if len(ch_names_all) < N_EEG_CHANNELS + N_EOG_CHANNELS:
        raise RuntimeError(
            f"Beklenen >= {N_EEG_CHANNELS + N_EOG_CHANNELS} kanal, "
            f"bulunan {len(ch_names_all)}"
        )
    eeg_names = ch_names_all[:N_EEG_CHANNELS]
    eog_names = ch_names_all[N_EEG_CHANNELS:N_EEG_CHANNELS + N_EOG_CHANNELS]

    ch_type_map = {name: "eeg" for name in eeg_names}
    ch_type_map.update({name: "eog" for name in eog_names})
    raw.set_channel_types(ch_type_map, verbose=mne_verbose)

    events, event_id_map = mne.events_from_annotations(raw, verbose=mne_verbose)
    wanted = CUE_EVENT_IDS_TRAIN if session.upper() == "T" else CUE_EVENT_ID_TEST

    selected_event_id: Dict[str, int] = {}
    for desc, label in wanted.items():
        if desc in event_id_map:
            selected_event_id[desc] = event_id_map[desc]
        elif session.upper() == "T":
            raise RuntimeError(f"Cue olayı '{desc}' GDF'de yok.")
    if not selected_event_id:
        raise RuntimeError("Hiçbir cue olayı bulunamadı.")

    epochs = mne.Epochs(
        raw, events=events, event_id=selected_event_id,
        tmin=tmin, tmax=tmax, baseline=None, preload=True, proj=False,
        picks=None, verbose=mne_verbose, on_missing="ignore",
    )

    eeg_data = epochs.copy().pick(eeg_names).get_data(copy=False)
    eog_data = epochs.copy().pick(eog_names).get_data(copy=False)
    eeg_data = eeg_data[:, :, :EPOCH_SAMPLES]
    eog_data = eog_data[:, :, :EPOCH_SAMPLES]

    if session.upper() == "T":
        inv_map = {v: CUE_EVENT_IDS_TRAIN[k] for k, v in selected_event_id.items()}
        raw_labels = np.array([inv_map[e] for e in epochs.events[:, 2]])
        y = raw_labels - LABEL_OFFSET
    else:
        mat_path = _find_true_labels_file(subject_id, data_dir)
        if mat_path is not None:
            y = _load_true_labels(mat_path)
            if len(y) != eeg_data.shape[0]:
                raise RuntimeError(
                    f"True label sayısı ({len(y)}) epoch sayısıyla "
                    f"({eeg_data.shape[0]}) eşleşmiyor."
                )
        else:
            y = None

    return SubjectData(
        X=eeg_data.astype(np.float32),
        y=y.astype(np.int64) if y is not None else None,
        X_eog=eog_data.astype(np.float32),
        subject_id=subject_id,
        session=session.upper(),
        sfreq=SFREQ,
        ch_names=eeg_names,
    )


def load_all_subjects(
    session: str = "T",
    data_dir: Optional[Path] = None,
    subjects: Optional[List[int]] = None,
    concatenate: bool = False,
    **kwargs,
):
    """Tüm denekleri yükle. concatenate=True ise (X, y, subject_ids) döner."""
    if subjects is None:
        subjects = list(range(1, 10))
    per_subject: Dict[int, SubjectData] = {}
    for sid in subjects:
        per_subject[sid] = load_subject(sid, session=session, data_dir=data_dir, **kwargs)
    if not concatenate:
        return per_subject
    if any(sd.y is None for sd in per_subject.values()):
        raise RuntimeError("concatenate=True session='E' ile kullanılamaz.")
    X_list, y_list, sid_list = [], [], []
    for sid, sd in per_subject.items():
        X_list.append(sd.X)
        y_list.append(sd.y)
        sid_list.append(np.full(sd.n_trials, sid, dtype=np.int64))
    return (np.concatenate(X_list, 0), np.concatenate(y_list, 0),
            np.concatenate(sid_list, 0))


def summarize(sd: SubjectData) -> str:
    lines = [
        f"Subject A{sd.subject_id:02d}{sd.session}",
        f"  X shape       : {sd.X.shape}  (trials, channels, samples)",
        f"  sfreq         : {sd.sfreq} Hz",
        f"  n_eeg_channels: {len(sd.ch_names)}",
    ]
    if sd.y is not None:
        lines.append(f"  y shape       : {sd.y.shape}")
        lines.append(f"  class dist    : {sd.class_distribution}")
    else:
        lines.append("  y             : None (true labels not loaded)")
    return "\n".join(lines)


# =========================================================================== #
# DEĞERLENDİRME  (evaluation.py)                                              #
# =========================================================================== #

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import (accuracy_score, cohen_kappa_score,
                             confusion_matrix, f1_score)
from sklearn.model_selection import GridSearchCV, StratifiedKFold


@dataclass
class CVResult:
    kappa: float
    accuracy: float
    macro_f1: float
    fold_kappas: List[float]
    confusion: np.ndarray
    y_true: np.ndarray
    y_pred: np.ndarray
    y_proba: Optional[np.ndarray] = None
    extras: Dict[str, Any] = field(default_factory=dict)

    def summary(self) -> str:
        return (f"kappa={self.kappa:.4f}  acc={self.accuracy:.4f}  "
                f"macro_f1={self.macro_f1:.4f}  "
                f"fold_kappa_std={np.std(self.fold_kappas):.4f}")


def compute_metrics(y_true, y_pred, labels=None) -> Dict[str, Any]:
    return {
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", labels=labels)),
        "confusion": confusion_matrix(y_true, y_pred, labels=labels),
    }


def _supports_proba(estimator) -> bool:
    return hasattr(estimator, "predict_proba")


def run_cv(estimator_factory, X, y, n_splits=5, shuffle=True,
           random_state=RANDOM_STATE, return_proba=True) -> CVResult:
    """Standart stratified k-fold CV; OOF tahminleri toplar."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
    n = len(y)
    oof_pred = np.full(n, -1, dtype=np.int64)
    oof_proba = None
    fold_kappas: List[float] = []
    for tr, te in skf.split(X, y):
        est = estimator_factory()
        est.fit(X[tr], y[tr])
        pred = est.predict(X[te])
        oof_pred[te] = pred
        fold_kappas.append(cohen_kappa_score(y[te], pred))
        if return_proba and _supports_proba(est):
            proba = est.predict_proba(X[te])
            if oof_proba is None:
                oof_proba = np.zeros((n, proba.shape[1]), dtype=np.float32)
            oof_proba[te] = proba
    m = compute_metrics(y, oof_pred)
    return CVResult(m["kappa"], m["accuracy"], m["macro_f1"], fold_kappas,
                    m["confusion"], y.copy(), oof_pred, oof_proba)


def run_nested_cv(estimator_factory, param_grid, X, y, outer_splits=5,
                  inner_splits=5, scoring="accuracy", random_state=RANDOM_STATE,
                  n_jobs=-1, return_proba=True) -> CVResult:
    """Nested CV: dış fold performans, iç fold (GridSearchCV) hiperparametre."""
    outer = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=random_state)
    inner = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=random_state)
    n = len(y)
    oof_pred = np.full(n, -1, dtype=np.int64)
    oof_proba = None
    fold_kappas: List[float] = []
    best_params: List[Dict[str, Any]] = []
    for tr, te in outer.split(X, y):
        gs = GridSearchCV(estimator=estimator_factory(), param_grid=param_grid,
                          cv=inner, scoring=scoring, n_jobs=n_jobs, refit=True)
        gs.fit(X[tr], y[tr])
        best_params.append(gs.best_params_)
        pred = gs.predict(X[te])
        oof_pred[te] = pred
        fold_kappas.append(cohen_kappa_score(y[te], pred))
        if return_proba and _supports_proba(gs.best_estimator_):
            proba = gs.predict_proba(X[te])
            if oof_proba is None:
                oof_proba = np.zeros((n, proba.shape[1]), dtype=np.float32)
            oof_proba[te] = proba
    m = compute_metrics(y, oof_pred)
    return CVResult(m["kappa"], m["accuracy"], m["macro_f1"], fold_kappas,
                    m["confusion"], y.copy(), oof_pred, oof_proba,
                    extras={"best_params_per_fold": best_params})


def soft_vote(proba_list: Sequence[np.ndarray],
              weights: Optional[Sequence[float]] = None) -> np.ndarray:
    """Ağırlıklı ortalama olasılık -> argmax."""
    if not proba_list:
        raise ValueError("proba_list boş olamaz.")
    stacked = np.stack(proba_list, axis=0)
    if weights is None:
        w = np.ones(stacked.shape[0]) / stacked.shape[0]
    else:
        w = np.asarray(weights, dtype=np.float64)
        w = w / w.sum()
    avg = np.tensordot(w, stacked, axes=([0], [0]))
    return avg.argmax(axis=1)


@dataclass
class SubjectRunResult:
    subject_id: int
    kappa: float
    accuracy: float
    macro_f1: float
    kappa_std: float
    n_trials: int
    elapsed_sec: float
    best_params_per_fold: List[Dict[str, Any]] = field(default_factory=list)


# =========================================================================== #
# FİLTRE BANKASI + FBCSP YAPI TAŞLARI  (_fbcsp_common.py)                     #
# =========================================================================== #

from scipy.signal import butter, filtfilt
from sklearn.feature_selection import (mutual_info_classif,
                                       mutual_info_regression)
from sklearn.utils.validation import check_random_state


def generate_bands(f_low=8.0, f_high=30.0,
                   widths_steps=((4.0, 2.0), (6.0, 3.0))) -> List[Tuple[float, float]]:
    """16 bant: (4Hz/adim2 -> 10 bant) + (6Hz/adim3 -> 6 bant)."""
    bands: List[Tuple[float, float]] = []
    for width, step in widths_steps:
        f = f_low
        while f + width <= f_high + 1e-9:
            bands.append((round(f, 4), round(f + width, 4)))
            f += step
    return bands


def _design_bandpass(l_freq, h_freq, sfreq, order=4):
    nyq = 0.5 * sfreq
    return butter(order, [l_freq / nyq, h_freq / nyq], btype="band")


def make_multiband_tensor(X, sfreq, bands, order=4) -> np.ndarray:
    """Veri-bağımsız filtreleme -> (n_trials, n_bands, n_channels, n_samples)."""
    n_trials, n_channels, n_samples = X.shape
    out = np.empty((n_trials, len(bands), n_channels, n_samples), dtype=np.float32)
    for bi, (l, h) in enumerate(bands):
        b, a = _design_bandpass(l, h, sfreq, order=order)
        out[:, bi] = filtfilt(b, a, X, axis=-1).astype(np.float32, copy=False)
    return out


class MultiBandCSP(BaseEstimator, TransformerMixin):
    """Her banda CSP (ledoit_wolf, log-var); feature'ları concat eder. 4D girdi."""

    def __init__(self, n_components=4, reg="ledoit_wolf", log=True, norm_trace=False):
        self.n_components = n_components
        self.reg = reg
        self.log = log
        self.norm_trace = norm_trace

    def fit(self, X, y):
        from mne.decoding import CSP
        if X.ndim != 4:
            raise ValueError(f"MultiBandCSP 4D bekler, aldı: {X.shape}")
        self.n_bands_ = X.shape[1]
        self.csps_ = []
        for bi in range(self.n_bands_):
            csp = CSP(n_components=self.n_components, reg=self.reg, log=self.log,
                      norm_trace=self.norm_trace, transform_into="average_power")
            csp.fit(X[:, bi].astype(np.float64, copy=False), y)
            self.csps_.append(csp)
        return self

    def transform(self, X):
        feats = [csp.transform(X[:, bi].astype(np.float64, copy=False))
                 for bi, csp in enumerate(self.csps_)]
        return np.concatenate(feats, axis=1).astype(np.float32, copy=False)


class AllBandCSP(BaseEstimator, TransformerMixin):
    """Her banda CSP, (n_trials, n_bands, n_components) tensörü döndürür.
    Sabit parametre -> Pipeline(memory=...) ile cache'lenebilir."""

    def __init__(self, n_components=4, reg="ledoit_wolf", log=True, norm_trace=False):
        self.n_components = n_components
        self.reg = reg
        self.log = log
        self.norm_trace = norm_trace

    def fit(self, X, y):
        from mne.decoding import CSP
        if X.ndim != 4:
            raise ValueError(f"AllBandCSP 4D bekler, aldı: {X.shape}")
        self.n_bands_ = X.shape[1]
        self.csps_ = []
        for bi in range(self.n_bands_):
            csp = CSP(n_components=self.n_components, reg=self.reg, log=self.log,
                      norm_trace=self.norm_trace, transform_into="average_power")
            csp.fit(X[:, bi].astype(np.float64, copy=False), y)
            self.csps_.append(csp)
        return self

    def transform(self, X):
        feats = np.stack([csp.transform(X[:, bi].astype(np.float64, copy=False))
                          for bi, csp in enumerate(self.csps_)], axis=1)
        return feats.astype(np.float32, copy=False)


class TopKBandSelector(BaseEstimator, TransformerMixin):
    """Her bandı MI ile skorla, en iyi top_k bandı seç (TRAIN-ONLY).
    Girdi (n, n_bands, n_comp) -> Çıktı (n, top_k*n_comp)."""

    def __init__(self, top_k=8, random_state=RANDOM_STATE):
        self.top_k = top_k
        self.random_state = random_state

    def fit(self, X, y):
        if X.ndim != 3:
            raise ValueError(f"TopKBandSelector 3D bekler, aldı: {X.shape}")
        n_trials, n_bands, n_comp = X.shape
        scores = np.empty(n_bands, dtype=np.float64)
        for bi in range(n_bands):
            mi = mutual_info_classif(X[:, bi, :], y,
                                     random_state=self.random_state, n_neighbors=3)
            scores[bi] = float(mi.sum())
        self.band_scores_ = scores
        k = min(self.top_k, n_bands)
        self.selected_bands_ = np.argsort(-scores)[:k]
        return self

    def transform(self, X):
        sel = X[:, self.selected_bands_, :]
        return sel.reshape(sel.shape[0], -1)


class MRMRSelector(BaseEstimator, TransformerMixin):
    """mRMR (MID): relevance MI(f,y) - ortalama redundancy MI(f, secilen)."""

    def __init__(self, n_features=20, random_state=RANDOM_STATE):
        self.n_features = n_features
        self.random_state = random_state

    def fit(self, X, y):
        _ = check_random_state(self.random_state)
        n_total = X.shape[1]
        k = min(self.n_features, n_total)
        rel = mutual_info_classif(X, y, random_state=self.random_state, n_neighbors=3)
        red_sum = np.zeros(n_total, dtype=np.float64)
        not_sel = np.ones(n_total, dtype=bool)
        first = int(np.argmax(rel))
        selected = [first]
        not_sel[first] = False
        if k > 1 and not_sel.any():
            idx = np.where(not_sel)[0]
            red = mutual_info_regression(X[:, idx], X[:, first],
                                         random_state=self.random_state, n_neighbors=3)
            red_sum[idx] += red
        while len(selected) < k:
            idx = np.where(not_sel)[0]
            scores = rel[idx] - red_sum[idx] / len(selected)
            best = int(idx[int(np.argmax(scores))])
            selected.append(best)
            not_sel[best] = False
            if not_sel.any() and len(selected) < k:
                idx = np.where(not_sel)[0]
                red = mutual_info_regression(X[:, idx], X[:, best],
                                             random_state=self.random_state, n_neighbors=3)
                red_sum[idx] += red
        self.selected_ = np.array(selected, dtype=np.int64)
        return self

    def transform(self, X):
        return X[:, self.selected_]


# =========================================================================== #
# RIEMANN YAPI TAŞLARI  (_riemann_common.py)                                  #
# =========================================================================== #


def bandpass_single(X, sfreq, l_freq=8.0, h_freq=30.0, order=4) -> np.ndarray:
    """Tek geniş bant 8-30 Hz Butterworth (filtfilt)."""
    nyq = 0.5 * sfreq
    b, a = butter(order, [l_freq / nyq, h_freq / nyq], btype="band")
    return filtfilt(b, a, X, axis=-1).astype(np.float32, copy=False)


def bandpass_multi(X, sfreq, bands, order=4) -> np.ndarray:
    """Her bant ayrı kopya: (n_trials, n_bands, n_ch, n_samp)."""
    out = np.empty((X.shape[0], len(bands), X.shape[1], X.shape[2]), dtype=np.float32)
    nyq = 0.5 * sfreq
    for bi, (l, h) in enumerate(bands):
        b, a = butter(order, [l / nyq, h / nyq], btype="band")
        out[:, bi] = filtfilt(b, a, X, axis=-1).astype(np.float32, copy=False)
    return out


class MultiBandTangentSpace(BaseEstimator, TransformerMixin):
    """Bant başına Covariances(oas) + TangentSpace(riemann), sonra concat."""

    def __init__(self, cov_estimator="oas", ts_metric="riemann"):
        self.cov_estimator = cov_estimator
        self.ts_metric = ts_metric

    def fit(self, X, y=None):
        from pyriemann.estimation import Covariances
        from pyriemann.tangentspace import TangentSpace
        if X.ndim != 4:
            raise ValueError(f"MultiBandTangentSpace 4D bekler, aldı: {X.shape}")
        self.n_bands_ = X.shape[1]
        self.covs_, self.tss_ = [], []
        for bi in range(self.n_bands_):
            cov = Covariances(estimator=self.cov_estimator)
            X_b = X[:, bi].astype(np.float64, copy=False)
            cov.fit(X_b)
            cov_X = cov.transform(X_b)
            ts = TangentSpace(metric=self.ts_metric)
            ts.fit(cov_X)
            self.covs_.append(cov)
            self.tss_.append(ts)
        return self

    def transform(self, X):
        feats = []
        for bi in range(self.n_bands_):
            X_b = X[:, bi].astype(np.float64, copy=False)
            cov_X = self.covs_[bi].transform(X_b)
            feats.append(self.tss_[bi].transform(cov_X))
        return np.concatenate(feats, axis=1).astype(np.float32, copy=False)


# =========================================================================== #
# DERİN ÖĞRENME ALTYAPISI  (_dl_common.py)                                    #
# =========================================================================== #

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int = RANDOM_STATE) -> None:
    """Tüm stochastic kanalları aynı tohuma bağlar."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Determinizmi artır (T4 üzerinde runlar arası varyansı azaltır)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def bandpass_wide(X, sfreq, l_freq=4.0, h_freq=38.0, order=4) -> np.ndarray:
    """4-38 Hz geniş bant (DL kendi temporal filtresini öğrenir)."""
    nyq = 0.5 * sfreq
    b, a = butter(order, [l_freq / nyq, h_freq / nyq], btype="band")
    return filtfilt(b, a, X, axis=-1).astype(np.float32, copy=False)


class EEGDataset(Dataset):
    """numpy (N, C, T) -> tensor (1, C, T) + label. Augmentation sadece train'de."""

    def __init__(self, X, y, augment=False, shift_max=12, noise_std=0.1):
        assert X.ndim == 3, f"X (N, C, T) olmalı, alındı {X.shape}"
        self.X = X.astype(np.float32, copy=False)
        self.y = y.astype(np.int64, copy=False)
        self.augment = augment
        self.shift_max = shift_max
        self.noise_std = noise_std
        self._ch_std = X.std(axis=(0, 2), keepdims=True) + 1e-12

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].copy()
        if self.augment:
            if self.shift_max > 0:
                shift = np.random.randint(-self.shift_max, self.shift_max + 1)
                if shift != 0:
                    x = np.roll(x, shift, axis=-1)
            if self.noise_std > 0:
                std = self._ch_std[0]
                x = x + np.random.randn(*x.shape).astype(np.float32) * (std * self.noise_std)
        x = x[np.newaxis, :, :]
        return torch.from_numpy(x), int(self.y[idx])


@dataclass
class TrainConfig:
    epochs: int = 300
    batch_size: int = 64
    lr: float = 1e-3
    weight_decay: float = 1e-2
    early_stopping_patience: int = 30
    val_fraction: float = 0.2
    augment: bool = True
    shift_max: int = 12
    noise_std: float = 0.1
    verbose: bool = False


def _stratified_inner_split(y, val_fraction, seed):
    n_splits = max(int(round(1.0 / val_fraction)), 2)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    tr_idx, va_idx = next(skf.split(np.zeros(len(y)), y))
    return tr_idx, va_idx


def _train_loop(model, dl_tr, dl_va, dl_te, cfg_epochs, lr, weight_decay, patience):
    """Ortak eğitim döngüsü: early stopping + en iyi state ile test predict_proba."""
    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    best_val_loss = float("inf")
    best_state = None
    best_epoch = -1
    pat = 0
    epoch = 0
    for epoch in range(cfg_epochs):
        model.train()
        for xb, yb in dl_tr:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            optim.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            optim.step()
        model.eval()
        vs, vn = 0.0, 0
        with torch.no_grad():
            for xb, yb in dl_va:
                xb = xb.to(DEVICE, non_blocking=True)
                yb = yb.to(DEVICE, non_blocking=True)
                vs += criterion(model(xb), yb).item() * xb.size(0)
                vn += xb.size(0)
        val_loss = vs / max(vn, 1)
        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    probas = []
    with torch.no_grad():
        for xb, _ in dl_te:
            xb = xb.to(DEVICE, non_blocking=True)
            probas.append(torch.softmax(model(xb), dim=-1).cpu().numpy())
    y_proba = np.concatenate(probas, axis=0)
    info = {"best_epoch": best_epoch, "best_val_loss": float(best_val_loss),
            "n_epochs_trained": epoch + 1, "best_state": best_state}
    return y_proba, info


def train_one_fold(model_factory, X_tr_outer, y_tr_outer, X_te_outer, cfg,
                   inner_seed=RANDOM_STATE):
    """Tek dış-fold: iç train/val (early stop) + dış-test predict_proba."""
    set_seed(inner_seed)
    tr_idx, va_idx = _stratified_inner_split(y_tr_outer, cfg.val_fraction, inner_seed)
    X_tr, y_tr = X_tr_outer[tr_idx], y_tr_outer[tr_idx]
    X_va, y_va = X_tr_outer[va_idx], y_tr_outer[va_idx]
    ds_tr = EEGDataset(X_tr, y_tr, augment=cfg.augment,
                       shift_max=cfg.shift_max, noise_std=cfg.noise_std)
    ds_va = EEGDataset(X_va, y_va, augment=False)
    ds_te = EEGDataset(X_te_outer, np.zeros(len(X_te_outer), dtype=np.int64), augment=False)
    dl_tr = DataLoader(ds_tr, batch_size=cfg.batch_size, shuffle=True, drop_last=False)
    dl_va = DataLoader(ds_va, batch_size=256, shuffle=False)
    dl_te = DataLoader(ds_te, batch_size=256, shuffle=False)
    model = model_factory().to(DEVICE)
    y_proba, info = _train_loop(model, dl_tr, dl_va, dl_te, cfg.epochs, cfg.lr,
                                cfg.weight_decay, cfg.early_stopping_patience)
    info.pop("best_state", None)
    return y_proba, info


def run_dl_subject(subject_id, model_factory, cfg, oof_store, oof_subdir,
                   outer_splits=5, verbose=True):
    """Tek denek 5-fold outer CV + OOF kaydet (oof_store: dict-of-dict)."""
    t0 = time.time()
    sd = load_subject(subject_id=subject_id, session="T", verbose=False)
    if sd.y is None:
        raise RuntimeError(f"A{subject_id:02d}T için etiket yok.")
    X = bandpass_wide(sd.X, sd.sfreq) * 1e6
    y = sd.y
    if verbose:
        print(f"[A{subject_id:02d}T] X={X.shape}, device={DEVICE}", flush=True)
    skf = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=RANDOM_STATE)
    n = len(y)
    n_classes = int(np.max(y) + 1)
    oof_pred = np.full(n, -1, dtype=np.int64)
    oof_proba = np.zeros((n, n_classes), dtype=np.float32)
    fold_kappas: List[float] = []
    for fi, (tr, te) in enumerate(skf.split(X, y)):
        inner_seed = RANDOM_STATE + fi
        y_proba_te, info = train_one_fold(model_factory, X[tr], y[tr], X[te],
                                          cfg, inner_seed=inner_seed)
        oof_proba[te] = y_proba_te
        oof_pred[te] = y_proba_te.argmax(axis=1)
        k = cohen_kappa_score(y[te], oof_pred[te])
        fold_kappas.append(k)
        if verbose:
            print(f"  fold {fi}: kappa={k:.4f}  best_epoch={info['best_epoch']}",
                  flush=True)
    kappa = cohen_kappa_score(y, oof_pred)
    acc = accuracy_score(y, oof_pred)
    mf1 = f1_score(y, oof_pred, average="macro")
    oof_store.setdefault(oof_subdir, {})[subject_id] = {
        "y_true": y.copy(), "y_pred": oof_pred.copy(), "y_proba": oof_proba.copy()}
    elapsed = time.time() - t0
    if verbose:
        print(f"[A{subject_id:02d}T] kappa={kappa:.4f}  acc={acc:.4f}  "
              f"elapsed={elapsed:.1f}s", flush=True)
    return SubjectRunResult(subject_id, kappa, acc, mf1, float(np.std(fold_kappas)),
                            n, elapsed)


# ----- Cross-subject transfer (Faz 5b) ------------------------------------- #


@dataclass
class TransferConfig:
    pre_epochs: int = 200
    pre_batch_size: int = 64
    pre_lr: float = 1e-3
    pre_weight_decay: float = 1e-2
    pre_patience: int = 30
    pre_val_fraction: float = 0.15
    ft_epochs: int = 100
    ft_batch_size: int = 64
    ft_lr: float = 1e-4
    ft_weight_decay: float = 1e-2
    ft_patience: int = 15
    ft_val_fraction: float = 0.2
    augment: bool = True
    shift_max: int = 12
    noise_std: float = 0.1
    verbose: bool = False


def _load_pretrain_pool(target_subject_id, all_subjects=tuple(range(1, 10))):
    pool = [s for s in all_subjects if s != target_subject_id]
    per_subj = load_all_subjects(session="T", subjects=pool, concatenate=False)
    Xs, ys = [], []
    for sid, sd in per_subj.items():
        if sd.y is None:
            raise RuntimeError(f"A{sid:02d}T etiketi yok.")
        Xs.append(bandpass_wide(sd.X, sd.sfreq) * 1e6)
        ys.append(sd.y)
    return (np.concatenate(Xs, 0).astype(np.float32),
            np.concatenate(ys, 0).astype(np.int64))


def _pretrain_model(model_factory, X, y, cfg, seed=RANDOM_STATE):
    set_seed(seed)
    tr_idx, va_idx = _stratified_inner_split(y, cfg.pre_val_fraction, seed)
    ds_tr = EEGDataset(X[tr_idx], y[tr_idx], augment=cfg.augment,
                       shift_max=cfg.shift_max, noise_std=cfg.noise_std)
    ds_va = EEGDataset(X[va_idx], y[va_idx], augment=False)
    dl_tr = DataLoader(ds_tr, batch_size=cfg.pre_batch_size, shuffle=True, drop_last=False)
    dl_va = DataLoader(ds_va, batch_size=256, shuffle=False)
    # Pretrain'de test gerekmez; küçük bir kukla loader kullan.
    ds_te = EEGDataset(X[va_idx][:2], y[va_idx][:2], augment=False)
    dl_te = DataLoader(ds_te, batch_size=2, shuffle=False)
    model = model_factory().to(DEVICE)
    _, info = _train_loop(model, dl_tr, dl_va, dl_te, cfg.pre_epochs, cfg.pre_lr,
                          cfg.pre_weight_decay, cfg.pre_patience)
    best_state = info.pop("best_state")
    return best_state, {"pretrain_best_epoch": info["best_epoch"],
                        "pretrain_n_samples": len(y)}


def _finetune_and_predict(pretrained_state, model_factory, X_tr_outer, y_tr_outer,
                          X_te_outer, cfg, inner_seed):
    set_seed(inner_seed)
    tr_idx, va_idx = _stratified_inner_split(y_tr_outer, cfg.ft_val_fraction, inner_seed)
    ds_tr = EEGDataset(X_tr_outer[tr_idx], y_tr_outer[tr_idx], augment=cfg.augment,
                       shift_max=cfg.shift_max, noise_std=cfg.noise_std)
    ds_va = EEGDataset(X_tr_outer[va_idx], y_tr_outer[va_idx], augment=False)
    ds_te = EEGDataset(X_te_outer, np.zeros(len(X_te_outer), dtype=np.int64), augment=False)
    dl_tr = DataLoader(ds_tr, batch_size=cfg.ft_batch_size, shuffle=True, drop_last=False)
    dl_va = DataLoader(ds_va, batch_size=256, shuffle=False)
    dl_te = DataLoader(ds_te, batch_size=256, shuffle=False)
    model = model_factory().to(DEVICE)
    with torch.no_grad():
        dummy = torch.zeros(1, 1, X_tr_outer.shape[1], X_tr_outer.shape[2], device=DEVICE)
        _ = model(dummy)  # LazyLinear parametrelerini oluştur
    model.load_state_dict(pretrained_state, strict=True)
    y_proba, info = _train_loop(model, dl_tr, dl_va, dl_te, cfg.ft_epochs, cfg.ft_lr,
                                cfg.ft_weight_decay, cfg.ft_patience)
    info.pop("best_state", None)
    return y_proba, info


def run_transfer_subject(subject_id, model_factory, cfg, oof_store, oof_subdir,
                         outer_splits=5, verbose=True):
    """LOSO pretrain (1 kere) + 5 dış-fold finetune/predict. OOF kaydet."""
    t0 = time.time()
    sd = load_subject(subject_id=subject_id, session="T", verbose=False)
    if sd.y is None:
        raise RuntimeError(f"A{subject_id:02d}T etiketi yok.")
    X_target = bandpass_wide(sd.X, sd.sfreq) * 1e6
    y_target = sd.y
    X_pool, y_pool = _load_pretrain_pool(subject_id)
    if verbose:
        print(f"[A{subject_id:02d}T] target X={X_target.shape}  pool X={X_pool.shape}",
              flush=True)
    pretrained_state, pre_info = _pretrain_model(model_factory, X_pool, y_pool, cfg)
    if verbose:
        print(f"[A{subject_id:02d}T] pretrain best@{pre_info['pretrain_best_epoch']}",
              flush=True)
    skf = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=RANDOM_STATE)
    n = len(y_target)
    n_classes = int(np.max(y_target) + 1)
    oof_pred = np.full(n, -1, dtype=np.int64)
    oof_proba = np.zeros((n, n_classes), dtype=np.float32)
    fold_kappas: List[float] = []
    for fi, (tr, te) in enumerate(skf.split(X_target, y_target)):
        inner_seed = RANDOM_STATE + fi
        y_proba_te, info = _finetune_and_predict(pretrained_state, model_factory,
                                                 X_target[tr], y_target[tr],
                                                 X_target[te], cfg, inner_seed)
        oof_proba[te] = y_proba_te
        oof_pred[te] = y_proba_te.argmax(axis=1)
        k = cohen_kappa_score(y_target[te], oof_pred[te])
        fold_kappas.append(k)
        if verbose:
            print(f"  fold {fi}: kappa={k:.4f}", flush=True)
    kappa = cohen_kappa_score(y_target, oof_pred)
    acc = accuracy_score(y_target, oof_pred)
    mf1 = f1_score(y_target, oof_pred, average="macro")
    oof_store.setdefault(oof_subdir, {})[subject_id] = {
        "y_true": y_target.copy(), "y_pred": oof_pred.copy(), "y_proba": oof_proba.copy()}
    elapsed = time.time() - t0
    if verbose:
        print(f"[A{subject_id:02d}T] TRANSFER kappa={kappa:.4f}  elapsed={elapsed:.1f}s",
              flush=True)
    return SubjectRunResult(subject_id, kappa, acc, mf1, float(np.std(fold_kappas)),
                            n, elapsed)


# =========================================================================== #
# MODELLER  (exp_eegnet.py / exp_shallowconvnet.py)                           #
# =========================================================================== #


class EEGNet(nn.Module):
    """EEGNet-8,2 (Lawhern et al. 2018). 4-sınıf motor imagery."""

    def __init__(self, n_classes=4, n_chans=22, n_samples=500, F1=8, D=2, F2=16,
                 kernel_length=64, pool1=4, pool2=8, dropout=0.5):
        super().__init__()
        self.conv1 = nn.Conv2d(1, F1, (1, kernel_length),
                               padding=(0, kernel_length // 2), bias=False)
        self.bn1 = nn.BatchNorm2d(F1)
        self.depthwise = nn.Conv2d(F1, F1 * D, (n_chans, 1), groups=F1, bias=False)
        self.bn2 = nn.BatchNorm2d(F1 * D)
        self.pool1 = nn.AvgPool2d((1, pool1))
        self.drop1 = nn.Dropout(dropout)
        self.sep_depth = nn.Conv2d(F1 * D, F1 * D, (1, 16), padding=(0, 8),
                                   groups=F1 * D, bias=False)
        self.sep_point = nn.Conv2d(F1 * D, F2, (1, 1), bias=False)
        self.bn3 = nn.BatchNorm2d(F2)
        self.pool2 = nn.AvgPool2d((1, pool2))
        self.drop2 = nn.Dropout(dropout)
        self.classifier = nn.LazyLinear(n_classes)

    def forward(self, x):
        x = self.conv1(x); x = self.bn1(x)
        x = self.depthwise(x); x = self.bn2(x); x = F.elu(x)
        x = self.pool1(x); x = self.drop1(x)
        x = self.sep_depth(x); x = self.sep_point(x); x = self.bn3(x); x = F.elu(x)
        x = self.pool2(x); x = self.drop2(x)
        x = x.flatten(1)
        return self.classifier(x)


def build_eegnet() -> nn.Module:
    return EEGNet()


def _safe_log(x, eps=1e-6):
    return torch.log(torch.clamp(x, min=eps))


class ShallowConvNet(nn.Module):
    """ShallowConvNet (Schirrmeister et al. 2017). FBCSP'nin DL karşılığı."""

    def __init__(self, n_classes=4, n_chans=22, n_samples=500, n_filters_time=40,
                 n_filters_spat=40, filter_time_length=25, pool_time_length=75,
                 pool_time_stride=15, dropout=0.5):
        super().__init__()
        self.temporal = nn.Conv2d(1, n_filters_time, (1, filter_time_length), bias=True)
        self.spatial = nn.Conv2d(n_filters_time, n_filters_spat, (n_chans, 1), bias=False)
        self.bn = nn.BatchNorm2d(n_filters_spat)
        self.pool = nn.AvgPool2d((1, pool_time_length), stride=(1, pool_time_stride))
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.LazyLinear(n_classes)

    def forward(self, x):
        x = self.temporal(x)
        x = self.spatial(x)
        x = self.bn(x)
        x = x * x
        x = self.pool(x)
        x = _safe_log(x)
        x = self.drop(x)
        x = x.flatten(1)
        return self.classifier(x)


def build_shallowconvnet() -> nn.Module:
    return ShallowConvNet()

In [ ]:
# Modülü yükle ve veri dizinini bağla
import importlib
import bci_lib
importlib.reload(bci_lib)
from bci_lib import *          # load_subject, run_nested_cv, modeller, ...
import bci_lib

bci_lib.DATA_DIR = DATA_DIR_PATH   # load_subject bu global'i kullanır
set_seed(RANDOM_STATE)
print("bci_lib yüklendi.  DEVICE =", DEVICE)

### 1.3 Yükleme & doğrulama

`load_subject()` bir `.gdf` dosyasını okur, **cue + 0.5–2.5 s** penceresinde epoch'lar
(2 s × 250 Hz = **500 örnek**), 22 EEG kanalını ayırır ve etiketleri 0–3'e çeker:

- **A0XT** (training): etiketler `.gdf` annotation'larından (`769–772` → `1–4` → `0–3`).
- **A0XE** (test): etiketler ayrı `.mat`'ten (`classlabel` − 1).

Beklenen: `X` şekli **(288, 22, 500)**, sınıf dağılımı **72/72/72/72** (dengeli).

In [ ]:
sd = load_subject(1, "T")
print(summarize(sd))
print()
print("Beklenen: X=(288, 22, 500), sınıflar {0,1,2,3}, dağılım 72/72/72/72")
assert sd.X.shape == (288, 22, 500), "Beklenmeyen epoch şekli!"
assert sd.class_distribution == {0: 72, 1: 72, 2: 72, 3: 72}, "Dengesiz sınıf!"
print("✓ Doğrulama geçti.")

In [ ]:
# Tüm fazlarda paylaşılan ORTAK fold yapısı — OOF hizalamasının temeli
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("StratifiedKFold(5, shuffle=True, random_state=42) — A01T:")
for i, (tr, te) in enumerate(skf.split(sd.X, sd.y)):
    print(f"  fold {i}: train={len(tr)}  test={len(te)}  ilk 5 test idx={te[:5].tolist()}")
print("\nBu fold yapısı HER fazda aynıdır → tüm modellerin OOF tahminleri trial-by-trial hizalı.")

## BÖLÜM 2 — Değerlendirme Altyapısı

Ana metrik **Cohen's κ** (BCI Comp IV standart metriği; şans seviyesini düzeltir).

- **`run_nested_cv`** — dış 5-fold ile *dürüst performans tahmini*, iç 5-fold
  `GridSearchCV` ile *hiperparametre seçimi*. Dış-test fold'una hiçbir bilgi sızmaz.
  Her dış fold için OOF (out-of-fold) `predict_proba` toplanır.
- **OOF deposu** — her base modelin trial-bazlı olasılıkları `OOF[base][subject]`
  altında saklanır. Tüm base'ler aynı fold yapısını kullandığı için bu olasılıklar
  hizalıdır → soft-voting ensemble *leakage-free* olur.
- **`soft_vote`** — ağırlıklı ortalama olasılık → `argmax`.

Aşağıda OOF deposu ve faz koşum yardımcılarını tanımlıyoruz.

In [ ]:
from sklearn.metrics import cohen_kappa_score

OOF = {}              # OOF[base_name][subject_id] = {"y_true","y_pred","y_proba"}
WEAK = {2, 4, 5, 6}   # Faz 1'de ortaya çıkan "zayıf" denekler (düşük κ)
PHASE_KAPPA = {}      # faz adı -> mean κ (özet tablosu için)

def store_oof(base, sid, res):
    OOF.setdefault(base, {})[sid] = {
        "y_true": res.y_true, "y_pred": res.y_pred, "y_proba": res.y_proba}

def run_classical_phase(base_name, prep_fn, build_fn, grid,
                        subjects=None, n_jobs=-1, verbose=True):
    # Her denek için: yükle -> ön-işle -> nested CV -> OOF kaydet. mean κ döndür.
    subjects = subjects or SUBJECTS
    res_k, t0 = {}, time.time()
    for sid in subjects:
        sd = load_subject(sid, "T")
        X = prep_fn(sd); y = sd.y
        r = run_nested_cv(build_fn, grid, X, y, n_jobs=n_jobs)
        store_oof(base_name, sid, r)
        res_k[sid] = r.kappa
        if verbose:
            print(f"  A{sid:02d}: κ={r.kappa:.4f}  acc={r.accuracy:.4f}")
    mk = float(np.mean(list(res_k.values())))
    wk = float(np.mean([res_k[s] for s in subjects if s in WEAK])) if any(s in WEAK for s in subjects) else float("nan")
    print(f"[{base_name}] mean κ={mk:.4f}  (zayıf grup κ={wk:.4f})  süre={(time.time()-t0)/60:.1f} dk")
    return res_k, mk

def ensemble_kappa(bases, weights=None, subjects=None):
    # OOF olasılıklarından soft-vote -> denek-bazlı κ.
    subjects = subjects or SUBJECTS
    res = {}
    for sid in subjects:
        probas = [OOF[b][sid]["y_proba"] for b in bases]
        yt = OOF[bases[0]][sid]["y_true"]
        res[sid] = cohen_kappa_score(yt, soft_vote(probas, weights))
    return res

def compare(got, expected, label):
    print(f"\n➡️  {label}: elde edilen κ = {got:.4f}   |   beklenen ≈ {expected}")
print("Altyapı hazır.")

## BÖLÜM 3 — Faz 1: Baseline (Regularized FBCSP + Shrinkage-LDA)

**Filter Bank Common Spatial Patterns** — motor imagery'nin altın standardı.

**Pipeline:**
1. **16 bant** filtre bankası (8–30 Hz): 10×(4 Hz, adım 2) + 6×(6 Hz, adım 3).
2. Her bantta **regularized CSP** (`reg='ledoit_wolf'`, log-variance feature) — Ledoit-Wolf
   shrinkage küçük örneklemde kovaryans tahminini stabilize eder.
3. Tüm bant feature'ları concat → **`SelectKBest(mutual_info_classif)`**.
4. **Shrinkage-LDA** (`solver='lsqr', shrinkage='auto'`) — örtük boyut indirgemeli, sağlam.

**İç CV grid:** `csp_n_components ∈ {2,4,6}`, `select_k ∈ {10,20,40,'all'}`.

**Beklenen:** CV κ ≈ **0.625**. Belirgin **bimodal** dağılım: güçlü grup (A01,03,07,08,09)
κ≈0.80, zayıf grup (A02,04,05,06) κ≈0.41.

> ⏱️ En yavaş klasik faz: ~30–80 dk (16 bant × CSP × geniş grid × nested CV).

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.pipeline import Pipeline

FBCSP_BANDS = generate_bands()
print(f"Filtre bankası ({len(FBCSP_BANDS)} bant):", FBCSP_BANDS)

def prep_fbcsp(sd):
    # Veri-bağımsız çoklu-bant filtreleme (CV DIŞINDA → leakage yok)
    return make_multiband_tensor(sd.X, sd.sfreq, FBCSP_BANDS, order=4)

def build_fbcsp():
    mi = partial(mutual_info_classif, random_state=RANDOM_STATE)
    return Pipeline([
        ("csp",    MultiBandCSP(n_components=4, reg="ledoit_wolf")),
        ("select", SelectKBest(score_func=mi, k=20)),
        ("lda",    LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")),
    ])

GRID_FBCSP = {"csp__n_components": [2, 4, 6], "select__k": [10, 20, 40, "all"]}

k_fbcsp, m_fbcsp = run_classical_phase("fbcsp_rlda", prep_fbcsp, build_fbcsp, GRID_FBCSP)
PHASE_KAPPA["Faz1: FBCSP+sLDA"] = m_fbcsp
compare(m_fbcsp, 0.625, "Faz 1 (FBCSP + Shrinkage-LDA)")

## BÖLÜM 4 — Faz 2–3: Ablasyon (NEGATİF SONUÇLAR)

Bilimsel dürüstlük için denenen ama **baseline'ı geçemeyen** yaklaşımlar. Her birinin
*neden* başarısız olduğunu açıklıyoruz — negatif sonuçlar da bilgi taşır ve bazıları
ensemble için "farklı hata profili" sağlar.

| Varyant | κ | Sonuç |
|:--|:-:|:--|
| Faz 2a — hard band-sel + linSVM(MI) | ≈0.579 | ✗ band seçimi bilgi kaybı |
| Faz 2b — + mRMR | ≈0.585 | ✗ MI'ya göre marjinal |
| Faz 3a — Riemann TS+LR (tek-bant) | ≈0.530 | ✗ filtre bankası avantajı yok |
| Faz 3b — Riemann MDM | ≈0.464 | ✗ ayrımcı bilgi modellemiyor |
| Faz 3c-PCA — multi-band TS+PCA+L2 | ≈0.542 | ✗ boyut/örnek oranı kötü |
| Faz 3c-L1 — multi-band TS+L1 | ≈0.550 | ✗ ama **ensemble'da kullanılacak** |

### 4.1 Faz 2 — Train-only band selection + Linear SVM

**Hipotez:** 16 bandı zorla tutmaktansa train-fold'da MI ile en iyi `top_k` bandı seç.
**Sonuç:** başarısız. Shrinkage-LDA zaten örtük boyut indirgemesi yapıyor; *hard* band
selection bu yumuşak regularizasyondan daha agresif → bilgi kaybı. Güçlü deneklerde sinyal
birden çok banda dağılmış; zayıf deneklerde MI band skoru küçük örneklemde gürültüye duyarlı.

> ⏱️ Geniş grid (top_k × select_k × C). `joblib.Memory` ile `AllBandCSP` cache'lenir. ~30–50 dk.

In [ ]:
import tempfile
from joblib import Memory
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

MEM = Memory(tempfile.mkdtemp(prefix="bci_cache_"), verbose=0)

def build_bandsel_svm():
    mi = partial(mutual_info_classif, random_state=RANDOM_STATE, n_neighbors=3)
    return Pipeline([
        ("csp_all",      AllBandCSP(n_components=4, reg="ledoit_wolf")),
        ("select_bands", TopKBandSelector(top_k=8)),
        ("scaler",       StandardScaler()),
        ("select_feats", SelectKBest(score_func=mi, k=20)),
        ("svc",          SVC(kernel="linear", probability=True,
                             decision_function_shape="ovr", random_state=RANDOM_STATE)),
    ], memory=MEM)

GRID_SVM = {"select_bands__top_k": [4, 6, 8, 10],
            "select_feats__k": [20, 40, "all"],
            "svc__C": [0.01, 0.1, 1.0, 10.0]}

_, m_svm = run_classical_phase("fbcsp_bandsel_svm", prep_fbcsp, build_bandsel_svm, GRID_SVM)
PHASE_KAPPA["Faz2a: bandSel+SVM(MI)"] = m_svm
compare(m_svm, 0.579, "Faz 2a (band-sel + linSVM, MI)")

In [ ]:
def build_bandsel_mrmr():
    return Pipeline([
        ("csp_all",      AllBandCSP(n_components=4, reg="ledoit_wolf")),
        ("select_bands", TopKBandSelector(top_k=8)),
        ("scaler",       StandardScaler()),
        ("select_feats", MRMRSelector(n_features=20)),
        ("svc",          SVC(kernel="linear", probability=True,
                             decision_function_shape="ovr", random_state=RANDOM_STATE)),
    ], memory=MEM)

GRID_MRMR = {"select_bands__top_k": [4, 6, 8, 10],
             "select_feats__n_features": [20, 40, 60],
             "svc__C": [0.01, 0.1, 1.0, 10.0]}

_, m_mrmr = run_classical_phase("fbcsp_bandsel_mrmr", prep_fbcsp, build_bandsel_mrmr, GRID_MRMR)
PHASE_KAPPA["Faz2b: bandSel+SVM(mRMR)"] = m_mrmr
compare(m_mrmr, 0.585, "Faz 2b (band-sel + mRMR)")

### 4.2 Faz 3 — Riemann geometrisi

**Hipotez:** kovaryans manifoldunda küçük örneklemde kovaryans tahmini CSP'den sağlam
olabilir; zayıf grupta kazanç beklenir. **Sonuç:** doğrulanmadı — hiçbir varyant Faz 1'i geçmedi.

- **3a TS (tek-bant 8–30 Hz):** FBCSP'nin 16-bant avantajını yakalayamaz → *adil olmayan kıyas*.
- **3b MDM:** sınıf merkezlerine saf metrik mesafe; ayrımcı bilgiyi modellemez.
- **3c Multi-band TS (3 bant):** 759 feature / 288 trial → boyut/örnek oranı kötü; PCA/L1
  boyut indirgemesi bilgi kaybeder. **L1 varyantı yine de farklı hata profili taşıdığı için
  Faz 4 ensemble'ında kullanılacaktır.**

In [ ]:
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from pyriemann.classification import MDM
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA

# --- 3a: Tangent Space + LR (tek geniş bant 8-30 Hz) ---
def prep_ts1(sd):   return bandpass_single(sd.X, sd.sfreq)        # (n, 22, 500)
def build_ts1():
    return Pipeline([
        ("cov", Covariances(estimator="oas")),
        ("ts",  TangentSpace(metric="riemann")),
        ("scaler", StandardScaler()),
        ("lr",  LogisticRegression(max_iter=1000, solver="lbfgs", random_state=RANDOM_STATE)),
    ])
_, m_ts1 = run_classical_phase("riemann_ts_1band", prep_ts1, build_ts1, {"lr__C": [0.01, 0.1, 1.0, 10.0]})
PHASE_KAPPA["Faz3a: TS 1-band"] = m_ts1
compare(m_ts1, 0.530, "Faz 3a (TS + LR, tek bant)")

In [ ]:
# --- 3b: Minimum Distance to Mean (hiperparametresiz) ---
def build_mdm():
    return Pipeline([("cov", Covariances(estimator="oas")), ("mdm", MDM(metric="riemann"))])
_, m_mdm = run_classical_phase("riemann_mdm", prep_ts1, build_mdm, {"mdm__metric": ["riemann"]})
PHASE_KAPPA["Faz3b: MDM"] = m_mdm
compare(m_mdm, 0.464, "Faz 3b (MDM)")

In [ ]:
# --- 3c: Multi-band Tangent Space (3 bant) → PCA+L2 ve L1 varyantları ---
RIEMANN_BANDS = [(8.0, 14.0), (12.0, 20.0), (18.0, 30.0)]
def prep_mbts(sd):  return bandpass_multi(sd.X, sd.sfreq, RIEMANN_BANDS, order=4)

def build_mbts_pca():
    return Pipeline([
        ("mbts",   MultiBandTangentSpace(cov_estimator="oas", ts_metric="riemann")),
        ("scaler", StandardScaler()),
        ("pca",    PCA(n_components=50, random_state=RANDOM_STATE)),
        ("lr",     LogisticRegression(max_iter=2000, solver="lbfgs", random_state=RANDOM_STATE)),
    ], memory=MEM)
GRID_PCA = {"pca__n_components": [30, 50, 80], "lr__C": [0.01, 0.1, 1.0, 10.0]}
_, m_pca = run_classical_phase("riemann_multiband_ts_pca", prep_mbts, build_mbts_pca, GRID_PCA)
PHASE_KAPPA["Faz3c: MB-TS PCA"] = m_pca
compare(m_pca, 0.542, "Faz 3c-PCA (multi-band TS + PCA + L2-LR)")

def build_mbts_l1():
    return Pipeline([
        ("mbts",   MultiBandTangentSpace(cov_estimator="oas", ts_metric="riemann")),
        ("scaler", StandardScaler()),
        ("lr",     LogisticRegression(max_iter=4000, penalty="l1", solver="saga",
                                      random_state=RANDOM_STATE)),
    ], memory=MEM)
_, m_l1 = run_classical_phase("riemann_multiband_ts_l1", prep_mbts, build_mbts_l1, {"lr__C": [0.01, 0.1, 1.0, 10.0]})
PHASE_KAPPA["Faz3c: MB-TS L1"] = m_l1
compare(m_l1, 0.550, "Faz 3c-L1 (multi-band TS + L1-LR)  [ensemble'da kullanılacak]")

## BÖLÜM 5 — Faz 4: Klasik Ensemble (Soft Voting)

İki klasik base'in OOF `predict_proba`'larını **eşit ağırlıkla** soft-vote ediyoruz:

**FBCSP+sLDA  ⊕  Multi-band TS-L1**

Her iki base aynı `random_state=42` fold yapısını kullandığından OOF'lar trial-bazlı hizalı
→ **leakage-free** ensemble. Ağırlıkları OOF κ'sına göre seçmek leakage olurdu; **sabit eşit
ağırlık** kullanıyoruz.

**Beklenen:** κ ≈ **0.6425** (Faz 1'e göre +0.018; en çok güçlü grupta varyans azalması).

In [ ]:
k_faz4 = ensemble_kappa(["fbcsp_rlda", "riemann_multiband_ts_l1"])
m_faz4 = float(np.mean(list(k_faz4.values())))
PHASE_KAPPA["Faz4: klasik ensemble"] = m_faz4
for sid in SUBJECTS:
    print(f"  A{sid:02d}: κ={k_faz4[sid]:.4f}")
compare(m_faz4, 0.6425, "Faz 4 (FBCSP+sLDA ⊕ MB-TS-L1, eşit)")

## BÖLÜM 6 — Faz 5: Derin Öğrenme + Final Ensemble

**Hipotez:** DL, klasiklerden *farklı hata profili* çıkarır → ensemble'a 5./6. üye olunca
zayıf grupta klasiklerin kaçırdığı trial'ları yakalayabilir.

**Mimariler:**
- **EEGNet-8,2** (Lawhern 2018): temporal Conv → depthwise spatial → separable conv. ~2k param.
- **ShallowConvNet** (Schirrmeister 2017): FBCSP'nin DL karşılığı (temporal+spatial conv → square → log-pool).

**Protokol:** Girdi (B,1,22,500), bandpass **4–38 Hz** + µV ölçek. Dış CV yine
`StratifiedKFold(5, random_state=42)` → klasiklerle hizalı. İç %80/%20 stratified val,
**early stopping** (patience=30). **Augmentation sadece train:** time-shift ±12 örnek +
Gaussian gürültü (σ=0.1×kanal std). AdamW (lr=1e-3, wd=1e-2).

**Beklenen (tek başına, zayıf):** EEGNet κ≈0.398, ShallowConvNet κ≈0.397 — Faz 1'in çok
altında (288 trial subject-specific DL için sınırda). Ama **hata profili ortogonal** (Jaccard≈0.29).

> ⏱️ GPU'da ~7–10 dk/mimari. CPU'da çok daha yavaş.

In [ ]:
# EEGNet — subject-specific, 5-fold OOF
dl_cfg = TrainConfig(epochs=300, batch_size=64, lr=1e-3, weight_decay=1e-2,
                     early_stopping_patience=30, augment=True)
t0 = time.time()
res_eeg = {}
for sid in SUBJECTS:
    r = run_dl_subject(sid, build_eegnet, dl_cfg, OOF, "eegnet", verbose=True)
    res_eeg[sid] = r.kappa
m_eeg = float(np.mean(list(res_eeg.values())))
PHASE_KAPPA["Faz5a: EEGNet"] = m_eeg
print(f"\n[eegnet] mean κ={m_eeg:.4f}  süre={(time.time()-t0)/60:.1f} dk")
compare(m_eeg, 0.398, "Faz 5a (EEGNet, subject-specific)")

In [ ]:
# ShallowConvNet — subject-specific, 5-fold OOF
t0 = time.time()
res_scn = {}
for sid in SUBJECTS:
    r = run_dl_subject(sid, build_shallowconvnet, dl_cfg, OOF, "shallowconvnet", verbose=True)
    res_scn[sid] = r.kappa
m_scn = float(np.mean(list(res_scn.values())))
PHASE_KAPPA["Faz5a: ShallowConvNet"] = m_scn
print(f"\n[shallowconvnet] mean κ={m_scn:.4f}  süre={(time.time()-t0)/60:.1f} dk")
compare(m_scn, 0.397, "Faz 5a (ShallowConvNet, subject-specific)")

### 6.1 (Opsiyonel) Faz 5b — Cross-subject LOSO transfer

Her hedef denek için diğer 8 deneğin tüm A0XT'siyle pretrain + hedef fold'da fine-tune.
Zayıf-DL deneklerinde (A05, A06) belirgin kazanç sağlar ama **final ensemble lideri
değiştirmez** (Faz 5a 0.6553 tepede kalır). Varsayılan olarak kapalı (`RUN_TRANSFER=False`,
~50 dk). Açmak için Bölüm 0'da `RUN_TRANSFER=True` yapın.

In [ ]:
if RUN_TRANSFER:
    tcfg = TransferConfig(pre_epochs=200, ft_epochs=100, ft_lr=1e-4, ft_patience=15, augment=True)
    for sid in SUBJECTS:
        run_transfer_subject(sid, build_eegnet, tcfg, OOF, "eegnet_transfer", verbose=True)
    for sid in SUBJECTS:
        run_transfer_subject(sid, build_shallowconvnet, tcfg, OOF, "shallowconvnet_transfer", verbose=True)
    print("Transfer OOF üretildi.")
else:
    print("Faz 5b transfer atlandı (RUN_TRANSFER=False).")

### 6.2 FİNAL ENSEMBLE

**FBCSP+sLDA + MB-TS-L1 + EEGNet + ShallowConvNet**, ağırlık **1 / 1 / 0.5 / 0.5**.

Klasik base'lere tam ağırlık, DL'lere yarı ağırlık — DL tek başına zayıf olduğu için
(eşit ağırlıkta zayıflığı taşır), 0.5 *sweet spot*'tur.

**Beklenen:** CV κ ≈ **0.6553** (Faz 1'e göre +0.030; A04/A06/A09 belirgin kazanır).

In [ ]:
FINAL_BASES = ["fbcsp_rlda", "riemann_multiband_ts_l1", "eegnet", "shallowconvnet"]
FINAL_WEIGHTS = [1.0, 1.0, 0.5, 0.5]
k_final_cv = ensemble_kappa(FINAL_BASES, FINAL_WEIGHTS)
m_final_cv = float(np.mean(list(k_final_cv.values())))
PHASE_KAPPA["Faz5a: FINAL ensemble (CV)"] = m_final_cv
print("Final ensemble CV — denek bazlı:")
for sid in SUBJECTS:
    d = k_final_cv[sid] - k_fbcsp[sid]
    print(f"  A{sid:02d}: κ={k_final_cv[sid]:.4f}   (Faz1'e Δ={d:+.4f})")
compare(m_final_cv, 0.6553, "Final ensemble (CV, 1/1/0.5/0.5)")

## BÖLÜM 7 — Faz 6: A0XE Final Test (single-shot) 🔓

**Şimdiye kadar A0XE'ye HİÇ dokunulmadı.** Tüm tuning iç CV'de yapıldı. Şimdi tek-shot test:

**Protokol (her denek için):**
1. Final pipeline base'lerini **TÜM A0XT (288 trial)** ile fit et (CV yok):
   - Klasik base'ler: A0XT'de `GridSearchCV(5-fold)` → best params → refit.
   - DL base'ler: A0XT %80/%20 + early stopping + augmentation (train-only).
2. **A0XE'yi İLK ve TEK kez** oku → her base `predict_proba`.
3. Soft-vote **1/1/0.5/0.5** → final tahmin → A0XE etiketiyle κ.

**Beklenen:** **Test κ = 0.6137**, accuracy = 0.71, Δ(Test−CV) = −0.042.

### 7.1 Etiket doğrulama
A0XE.mat ham 1–4 → −1 kaydır → 0–3, A0XT semantiğiyle hizalı, 72/72/72/72 dengeli mi?

In [ ]:
from scipy.io import loadmat
for sid in (1, 5):
    raw = loadmat(str(LABELS_DIR / f"A{sid:02d}E.mat"))["classlabel"].ravel().astype(int)
    shifted = raw - 1
    uniq, cnt = np.unique(shifted, return_counts=True)
    print(f"A{sid:02d}E: n={len(raw)}  ham unique={sorted(set(raw))}  "
          f"kaydırılmış dağılım={dict(zip(uniq.tolist(), cnt.tolist()))}")
print("✓ A0XE etiketleri 0–3, dengeli 72/72/72/72 — A0XT ile aynı şema.")

### 7.2 Final değerlendirme

> ⏱️ ~25–35 dk (9 denek × [FBCSP grid + Riemann grid + 2 DL eğitimi]).

In [ ]:
from sklearn.model_selection import GridSearchCV
from bci_lib import _train_loop, _stratified_inner_split   # final DL fit için iç yardımcılar

def final_fit_classical(build_fn, grid, Xtr, ytr, Xte):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    gs = GridSearchCV(build_fn(), grid, cv=cv, scoring="accuracy", n_jobs=-1, refit=True)
    gs.fit(Xtr, ytr)
    return gs.predict_proba(Xte), gs.best_params_

def final_fit_dl(model_factory, Xtr, ytr, Xte, cfg):
    set_seed(RANDOM_STATE)
    from torch.utils.data import DataLoader
    tr_idx, va_idx = _stratified_inner_split(ytr, cfg.val_fraction, RANDOM_STATE)
    ds_tr = EEGDataset(Xtr[tr_idx], ytr[tr_idx], augment=cfg.augment,
                       shift_max=cfg.shift_max, noise_std=cfg.noise_std)
    ds_va = EEGDataset(Xtr[va_idx], ytr[va_idx], augment=False)
    ds_te = EEGDataset(Xte, np.zeros(len(Xte), dtype=np.int64), augment=False)
    dl_tr = DataLoader(ds_tr, batch_size=cfg.batch_size, shuffle=True)
    dl_va = DataLoader(ds_va, batch_size=256)
    dl_te = DataLoader(ds_te, batch_size=256)
    model = model_factory().to(DEVICE)
    y_proba, _ = _train_loop(model, dl_tr, dl_va, dl_te, cfg.epochs, cfg.lr,
                             cfg.weight_decay, cfg.early_stopping_patience)
    return y_proba

rows = []
y_true_all, y_pred_all = [], []   # birleşik confusion için
t0 = time.time()
for sid in SUBJECTS:
    sd_t = load_subject(sid, "T"); sd_e = load_subject(sid, "E")
    Xt, yt = sd_t.X, sd_t.y
    Xe, ye = sd_e.X, sd_e.y        # A0XE — İLK ve TEK kullanım

    # Base 1: FBCSP+sLDA
    p_fb, bp_fb = final_fit_classical(
        build_fbcsp, GRID_FBCSP,
        make_multiband_tensor(Xt, sd_t.sfreq, FBCSP_BANDS, 4),
        yt, make_multiband_tensor(Xe, sd_e.sfreq, FBCSP_BANDS, 4))
    # Base 2: Riemann MB-TS-L1
    p_rm, bp_rm = final_fit_classical(
        build_mbts_l1, {"lr__C": [0.01, 0.1, 1.0, 10.0]},
        bandpass_multi(Xt, sd_t.sfreq, RIEMANN_BANDS, 4),
        yt, bandpass_multi(Xe, sd_e.sfreq, RIEMANN_BANDS, 4))
    # Base 3-4: DL
    Xt_dl = bandpass_wide(Xt, sd_t.sfreq) * 1e6
    Xe_dl = bandpass_wide(Xe, sd_e.sfreq) * 1e6
    p_eg = final_fit_dl(build_eegnet, Xt_dl, yt, Xe_dl, dl_cfg)
    p_sc = final_fit_dl(build_shallowconvnet, Xt_dl, yt, Xe_dl, dl_cfg)

    pred_final = soft_vote([p_fb, p_rm, p_eg, p_sc], [1.0, 1.0, 0.5, 0.5])
    pred_faz1  = p_fb.argmax(1)
    k_test  = cohen_kappa_score(ye, pred_final)
    a_test  = (pred_final == ye).mean()
    k_cv    = k_final_cv.get(sid, float("nan"))
    rows.append({"sid": sid, "cv_kappa": k_cv, "test_kappa": k_test,
                 "test_acc": a_test, "faz1_test": cohen_kappa_score(ye, pred_faz1),
                 "delta": k_test - k_cv, "fbcsp_bp": bp_fb, "riemann_bp": bp_rm})
    y_true_all.append(ye); y_pred_all.append(pred_final)
    print(f"A{sid:02d}: Test κ={k_test:.4f}  acc={a_test:.4f}  "
          f"(CV κ={k_cv:.4f}, Δ={k_test-k_cv:+.4f})  FBCSP={bp_fb}")

y_true_all = np.concatenate(y_true_all); y_pred_all = np.concatenate(y_pred_all)
print(f"\nToplam süre: {(time.time()-t0)/60:.1f} dk")

In [ ]:
import pandas as pd
df = pd.DataFrame(rows).set_index("sid")
mean_test = df["test_kappa"].mean()
mean_cv   = df["cv_kappa"].mean()
mean_acc  = df["test_acc"].mean()
mean_d    = df["delta"].mean()

print("="*72)
print("FINAL DEĞERLENDİRME — A0XE saklı test (denek bazlı)")
print("="*72)
print(df[["cv_kappa", "test_kappa", "delta", "test_acc", "faz1_test"]].round(4).to_string())
print("-"*72)
print(f"MEAN  CV κ={mean_cv:.4f}   Test κ={mean_test:.4f}   "
      f"Δ(Test−CV)={mean_d:+.4f}   Test acc={mean_acc:.4f}")
print()
print(f"  Yarışma kazananı (Kai Keng Ang FBCSP, 2008):  κ = 0.569")
print(f"  Bu çalışma (final ensemble):                  κ = {mean_test:.4f}   "
      f"(Δ = {mean_test-0.569:+.4f})")
PHASE_KAPPA["Faz6: FINAL TEST (A0XE)"] = mean_test
compare(mean_test, "0.6137 (acc≈0.71, Δ≈-0.042)", "Faz 6 FINAL TEST")

## BÖLÜM 8 — Görselleştirme & Özet

1. Ablation bar grafiği (tüm fazların κ'sı)
2. CV vs Test denek-bazlı karşılaştırma
3. Final confusion matrix (normalize)
4. Özet tablo

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Ablation bar: tüm fazlar ---
order = ["Faz1: FBCSP+sLDA", "Faz2a: bandSel+SVM(MI)", "Faz2b: bandSel+SVM(mRMR)",
         "Faz3a: TS 1-band", "Faz3b: MDM", "Faz3c: MB-TS PCA", "Faz3c: MB-TS L1",
         "Faz4: klasik ensemble", "Faz5a: EEGNet", "Faz5a: ShallowConvNet",
         "Faz5a: FINAL ensemble (CV)", "Faz6: FINAL TEST (A0XE)"]
labels = [o for o in order if o in PHASE_KAPPA]
vals   = [PHASE_KAPPA[o] for o in labels]
colors = ["#2c7fb8" if "FINAL" in l or "Faz6" in l else
          ("#41ab5d" if "Faz4" in l else "#bdbdbd") for l in labels]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(range(len(vals)), vals, color=colors)
ax.axhline(0.569, ls="--", c="red", lw=1, label="Yarışma kazananı (0.569)")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Cohen's κ"); ax.set_title("Faz ilerlemesi — Cohen's κ")
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v+0.005, f"{v:.3f}", ha="center", fontsize=7)
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# --- 2. CV vs Test denek bazlı ---
sids = list(df.index)
x = np.arange(len(sids)); w = 0.38
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(x - w/2, df["cv_kappa"], w, label="CV (A0XT)", color="#9ecae1")
ax.bar(x + w/2, df["test_kappa"], w, label="Test (A0XE)", color="#3182bd")
ax.set_xticks(x); ax.set_xticklabels([f"A{s:02d}" for s in sids])
ax.set_ylabel("Cohen's κ")
ax.set_title(f"Final ensemble — CV κ={mean_cv:.4f} vs Test κ={mean_test:.4f}")
ax.axhline(0, c="k", lw=0.5); ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# --- 3. Final confusion matrix (normalize) ---
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true_all, y_pred_all)
cmn = cm.astype(float) / cm.sum(axis=1, keepdims=True)
class_names = ["sol el", "sağ el", "iki ayak", "dil"]
fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(class_names, rotation=30, ha="right"); ax.set_yticklabels(class_names)
ax.set_xlabel("Tahmin"); ax.set_ylabel("Gerçek")
ax.set_title("Final ensemble confusion (A0XE, normalize)")
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center",
                color="white" if cmn[i,j] > 0.5 else "black")
fig.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

In [ ]:
# --- 4. Özet tablo ---
print("="*60)
print(f"{'Faz / Pipeline':40s} {'κ':>8s}")
print("="*60)
for k in order:
    if k in PHASE_KAPPA:
        print(f"{k:40s} {PHASE_KAPPA[k]:8.4f}")
print("="*60)
print(f"{'Yarışma kazananı (Kai Keng Ang FBCSP)':40s} {0.569:8.4f}")
print(f"{'>>> FINAL Test κ (A0XE)':40s} {mean_test:8.4f}")
print(f"{'>>> FINAL Test accuracy':40s} {mean_acc:8.4f}")
print("="*60)
print("\n🎉 Yeniden-üretim tamamlandı. Saklı test setinde dürüst Cohen's κ ölçüldü;")
print("   BCI Comp IV 2a yarışma kazananı +0.045 farkla geçildi.")